In [5]:
import pandas as pd
import numpy as np

In [6]:
!pip install sentence_transformers

In [7]:
!pip install tf-keras

In [8]:
from sentence_transformers import SentenceTransformer, util

In [9]:
df=pd.read_csv("train.csv")

df=df.dropna()


df= df.sample(frac=0.10, random_state=42) 

df.head()
# df.shape[0]


df_t=pd.read_csv("test.csv")
df_t=df_t.dropna()


cat_path=df['category_path']

org_qr=df['origin_query']
print(df.shape[0])
print(df_t.shape[0])

2131
1184


In [10]:
df.head()
# df.isna().sum()

,task,language,origin_query,category_path,label
15449,QC,it,carplay moto,"motorcycle equipments & parts,motorcycle acces...",1
10782,QC,de,kpop Schlüsselanhänger,"jewelry & accessories,fashion jewelry,key chains",1
9933,QC,ko,일본 남자,"novelty & special use,world apparel,traditiona...",0
6825,QC,es,protector de tarjeta,"office & school supplies,desk accessories & or...",0
14031,QC,it,qqlt,"apparel accessories,eyewear & accessories,sung...",0


In [11]:
cat_path = cat_path.apply(lambda x: x.split(",") if pd.notna(x) else [])

In [12]:
cat_path.dtype

dtype('O')

In [13]:
model = SentenceTransformer('sentence-transformers/all-MiniLM-L6-v2')

In [14]:

from sentence_transformers import SentenceTransformer, util
import numpy as np

model = SentenceTransformer('paraphrase-multilingual-MiniLM-L12-v2')

def calculate_product_score(product_name, tags, model=model):
    if not isinstance(product_name, str) or not tags or len(tags) == 0:
        return np.nan
    product_embedding = model.encode(product_name, convert_to_tensor=True)
    tag_embeddings = model.encode(tags, convert_to_tensor=True)
    similarities = [util.cos_sim(product_embedding, tag_emb)[0].item() for tag_emb in tag_embeddings]
    # Compute weights: 1, 0.5, 0.25, ...
    weights = [0.4** i for i in range(len(similarities))]
    weighted_mean = np.average(similarities, weights=weights)
    return weighted_mean

# Apply function row-wise
# df['Score'] = df.apply(lambda row: calculate_product_score(row['origin_query'], row['category_path']), axis=1)

In [15]:
# df['Score'] = df.apply(lambda row: calculate_product_score(row['origin_query'], row['category_path']), axis=1)
print(calculate_product_score("Shirt",["Clothing","Apparel","Cotton"]))
# print(calculate_product_score("Shirt",["Shirt","Apparel","Cotton"]))
df['score'] = [calculate_product_score(name, tags) for name, tags in zip(org_qr.values, cat_path.values)]

0.557737145668421


In [16]:
df.head()

,task,language,origin_query,category_path,label,score
15449,QC,it,carplay moto,"motorcycle equipments & parts,motorcycle acces...",1,0.574312
10782,QC,de,kpop Schlüsselanhänger,"jewelry & accessories,fashion jewelry,key chains",1,0.403361
9933,QC,ko,일본 남자,"novelty & special use,world apparel,traditiona...",0,0.271373
6825,QC,es,protector de tarjeta,"office & school supplies,desk accessories & or...",0,0.210521
14031,QC,it,qqlt,"apparel accessories,eyewear & accessories,sung...",0,0.285094


In [17]:
from sklearn.metrics import f1_score
from sklearn.metrics import precision_score, recall_score
from sklearn.metrics import confusion_matrix
df['result']=[0 if scr<0.20 else 1 for scr in df['score']]
df.head()

f1_sc=f1_score(df['label'],df['result'])

df['true'] = [1 if lbl == rslt else 0 for lbl, rslt in zip(df['label'].values, df['result'].values)]
df.head()
df['true'].sum()


# Precision
precision = precision_score(df['label'], df['result'])
print("Precision:", precision)

# Recall
recall = recall_score(df['label'], df['result'])
print("Recall:", recall)
cm = confusion_matrix(df['label'], df['result'])
print(cm)
print(f1_sc)

print(df['true'].sum())


from sklearn.metrics import classification_report

print(classification_report(df['label'],df['result']))

Precision: 0.6853348729792148
Recall: 0.8266016713091922
[[ 150  545]
 [ 249 1187]]
0.7493686868686869
1337
              precision    recall  f1-score   support

           0       0.38      0.22      0.27       695
           1       0.69      0.83      0.75      1436

    accuracy                           0.63      2131
   macro avg       0.53      0.52      0.51      2131
weighted avg       0.58      0.63      0.59      2131

